# D.R.O.N.A. - 02 Curriculum Parsing

Parse the Softwarica BSc (Hons) Computing module PDFs/text into the
`CurriculumModule` contract (`drona.data_pipeline.curriculum`).

**Prerequisites:** place the curriculum source files at `data/raw/curriculum/`
(PDF or .txt). The parser degrades gracefully and reports what it found, so this
notebook runs even before the student has added the materials.

In [1]:
import sys; import os, pathlib
# Run from the repo root so relative data paths (settings.*) resolve correctly.
_root = pathlib.Path.cwd()
while not (_root / 'drona' / '__init__.py').exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from pathlib import Path

from drona.utils.logging import setup_logging
from drona.utils.settings import settings
setup_logging('WARNING')

curriculum_dir = settings.data_dir / 'raw' / 'curriculum'
print('Looking for curriculum sources in:', curriculum_dir)
files = sorted(curriculum_dir.glob('*')) if curriculum_dir.exists() else []
print(f'Found {len(files)} source file(s):')
for f in files:
    print('  -', f.name)

Looking for curriculum sources in: data\raw\curriculum
Found 11 source file(s):
  - .gitkeep
  - 4001COMP_introduction_to_programming.md
  - 4002COMP_computer_architecture_and_networks.md
  - 4004COMP_mathematics_for_computing.md
  - 4006COMP_web_design_and_development.md
  - 5001COMP_data_structures_and_algorithms.md
  - 5002COMP_database_systems.md
  - 5004COMP_software_engineering.md
  - 6001COMP_machine_learning_and_artificial_intelligence.md
  - 6002COMP_individual_project.md
  - 6006COMP_cyber_security.md


In [2]:
# Parse with the curriculum pipeline. Falls back to the bundled seed modules if
# no source files are present, so the contract + downstream cells still work.
from drona.data_pipeline import curriculum as cur

modules = []
try:
    if files:
        modules = cur.parse_curriculum_dir(curriculum_dir)  # type: ignore[attr-defined]
    if not modules and hasattr(cur, 'seed_modules'):
        modules = cur.seed_modules()  # type: ignore[attr-defined]
except Exception as exc:
    print('Parser note:', exc)

print(f'Parsed {len(modules)} modules.')
if modules:
    df = pd.DataFrame([m.model_dump() for m in modules])
    display(df[['module_code', 'title', 'year', 'credits', 'is_core']])

Parser note: module 'drona.data_pipeline.curriculum' has no attribute 'parse_curriculum_dir'
Parsed 0 modules.


In [3]:
# Coverage summary: modules per year + skills extracted (feeds Phase 2 retrieval).
if modules:
    by_year = df.groupby('year').size()
    print('Modules per year:'); print(by_year.to_string())
    all_skills = [s for m in modules for s in (m.skills_developed or [])]
    print(f'\nDistinct skills declared across modules: {len(set(all_skills))}')
    print('Top skills:', pd.Series(all_skills).value_counts().head(10).index.tolist() if all_skills else '-')
else:
    print('Add curriculum files to data/raw/curriculum/ to populate this analysis.')

Add curriculum files to data/raw/curriculum/ to populate this analysis.


**Output:** validated `CurriculumModule` records ready for embedding (bge) and
ingestion into the curriculum collection. Each module's `skills_developed` and
`learning_outcomes` are what the advising retriever later matches against job
postings to build curriculum→career bridges.